# 公式チュートリアル 04 — How to Run a Simulation
> 出典: https://ecell4.e-cell.org/tutorials/tutorial04.html

**Simulator** が Model と World を受け取り、`step()` / `run()` で時間を進める。**Factory** パターンで
ソルバを差し替えられる（同じ関数で ode/gillespie/... を切り替え）。

In [1]:
import warnings; warnings.filterwarnings('ignore')
from ecell4.prelude import *

with reaction_rules():
    A + B > C | 0.01
    C > A + B | 0.3
m = get_model()

# 低レベル: World + Simulator を直接使う
w = gillespie.World(Real3(1,1,1)); w.bind_to(m); w.add_molecules(Species('C'), 60)
sim = gillespie.Simulator(w)
print('step(1.0) ->', sim.step(1.0), 'now t =', round(sim.t(), 3))
while sim.step(5.0): pass          # t=5 まで進める
print('after run to 5.0: C =', w.num_molecules(Species('C')))

# Factory パターン: ソルバを引数で差し替え
def singlerun(f, m):
    w = f.world(Real3(1,1,1)); w.bind_to(m); w.add_molecules(Species('C'), 60)
    sim = f.simulator(w); sim.run(10.0)
    return w.num_molecules(Species('A'))
for f in [ode.Factory(), gillespie.Factory()]:
    print(type(f).__module__, '-> A at t=10:', round(singlerun(f, m), 1))

step(1.0) -> True now t = 0.078
after run to 5.0: C = 31
ecell4_base.ode -> A at t=10: 29
ecell4_base.gillespie -> A at t=10: 34


**要点**: `sim.step(upto)` は 1 イベント進めて `upto` 未満なら True。`while sim.step(t): pass` で `t` まで、
`sim.run(tau)` で `tau` だけ進む。**World/Simulator は Model を変更しない**ので、1 つの Model に対して
複数の Simulator を差し替えられる（Factory パターン）。